# A01. Muestreo para Validación Psiquiátrica

**Objetivo:** generar muestra estructurada para validación clínica post-denoising.
**Rol en la tesis:** APÉNDICE.
**Entradas requeridas:**
- `data/splits/dataset_base.csv`.
- `data/splits/train_denoised.csv`.
**Salidas esperadas:**
- `data/VALIDACION_PSIQUIATRAS_POST_DENOISING.xlsx`.


In [ ]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Importar utilidades compartidas
current_dir = Path.cwd()
if current_dir.name == "notebooks":
    sys.path.append(str(current_dir.parent))
else:
    sys.path.append(str(current_dir))

try:
    from notebooks.utils_shared import setup_paths
except ImportError:
    sys.path.append(str(current_dir))
    from utils_shared import setup_paths

paths = setup_paths()
DATA_PATH = paths['DATA_PATH']
SPLITS_PATH = paths['SPLITS_PATH']

print(f"Rutas configuradas.")

## 1. Cargar Datos: Original vs Denoised


In [ ]:
# Cargar conjunto de datos base (todos los casos)
df_base = pd.read_csv(SPLITS_PATH / 'dataset_base.csv')

# Cargar conjunto de datos depurado (solo casos que pasaron el filtro)
try:
    df_denoised = pd.read_csv(SPLITS_PATH / 'train_denoised.csv')
    denoised_ids = set(df_denoised['row_id'])
except FileNotFoundError:
    print("Error: No se encontró train_denoised.csv. Ejecuta pipeline/03_denoising_reglas_core.ipynb primero.")
    raise

# Identificar excluidos
df_base['is_kept'] = df_base['row_id'].isin(denoised_ids)
df_excluded = df_base[~df_base['is_kept']].copy()
df_included = df_base[df_base['is_kept']].copy()

print(f"Total casos: {len(df_base)}")
print(f"Casos MANTENIDOS (Clinical Signal): {len(df_included)} ({len(df_included)/len(df_base):.1%})")
print(f"Casos EXCLUIDOS (Noise/No Signal): {len(df_excluded)} ({len(df_excluded)/len(df_base):.1%})")


## 2. Selección de Casos para Validación

Seleccionaremos 25 casos distribuidos estratégicamente:


In [ ]:
casos_seleccionados = []

# --- GRUPO 1: Excluidos sospechosos (posibles falsos negativos) ---
# Identificar excluidos con longitud > 50 caracteres; si fueron removidos,
# podrían contener variantes paraguayas no capturadas por reglas.
excluidos_largos = df_excluded[df_excluded['texto'].str.len() > 50]
if not excluidos_largos.empty:
    sample_excluidos = excluidos_largos.sample(min(10, len(excluidos_largos)), random_state=42)
    sample_excluidos['Grupo'] = 'Excluido_Sospechoso'
    sample_excluidos['Motivo'] = 'Texto largo pero borrado por denoising. ¿Contiene síntomas no detectados?'
    casos_seleccionados.append(sample_excluidos)

# --- GRUPO 2: Inclusiones con negación (validar criterio clínico) ---
# Identificar casos conservados que contengan "no" o "niega".
# Objetivo: confirmar si psiquiatría considera esta negación como señal clínica.
mask_negacion = df_included['texto'].str.lower().str.contains(r'\b(no|niega|sin)\b', regex=True, na=False)
incluidos_negados = df_included[mask_negacion]
if not incluidos_negados.empty:
    sample_negados = incluidos_negados.sample(min(8, len(incluidos_negados)), random_state=42)
    sample_negados['Grupo'] = 'Inclusion_Negacion'
    sample_negados['Motivo'] = 'Contiene negaciones. ¿Es síntoma (falta insight) o ausencia real?'
    casos_seleccionados.append(sample_negados)

# --- GRUPO 3: Casos cortos mantenidos (posibles falsos positivos) ---
# Revisar casos cortos que pasaron el filtro para estimar utilidad clínica.
incluidos_cortos = df_included[df_included['texto'].str.len() < 100]
if not incluidos_cortos.empty:
    sample_cortos = incluidos_cortos.sample(min(7, len(incluidos_cortos)), random_state=42)
    sample_cortos['Grupo'] = 'Inclusion_Corta'
    sample_cortos['Motivo'] = 'Texto corto mantenido. ¿Es suficiente evidencia?'
    casos_seleccionados.append(sample_cortos)

# Unificar muestras para revisión
df_validacion = pd.concat(casos_seleccionados).head(25)

# Preparar columnas para el Excel
cols_export = ['row_id', 'Grupo', 'etiqueta', 'texto', 'Motivo']
df_export = df_validacion[cols_export].copy()
df_export['Validacion_Psiquiatra'] = ''  # Espacio para que escriban
df_export['Comentarios'] = ''

print(f"Seleccionados {len(df_export)} casos para validación.")
print(df_export['Grupo'].value_counts())

## 3. Exportar a Excel


In [ ]:
output_file = DATA_PATH / "VALIDACION_PSIQUIATRAS_POST_DENOISING.xlsx"
try:
    df_export.to_excel(output_file, index=False)
    print(f"Excel generado: {output_file}")
except ImportError:
    print("Error: Falta openpyxl. Instalando...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
    df_export.to_excel(output_file, index=False)
    print(f"Excel generado: {output_file}")